# 04_tuning_optuna.ipynb

Objetivo:
- Optimizar hiperparámetros de XGBoost sobre los embeddings con Optuna.
- Usar StratifiedKFold dentro del objective para medir rendimiento robusto.
- Guardar el mejor conjunto de hiperparámetros y reentrenar un modelo final.


# Celda 2 — Instalar Optuna 

In [ ]:
# Ejecutar solo si no tienes optuna
%pip install optuna --quiet


# Celda 3 — Imports y carga de embeddings 

In [ ]:
import os
import joblib
import json
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import xgboost as xgb
import optuna
import random

# Paths (ajusta si tu estructura es distinta)
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
MODELS_DIR = os.path.join(BASE_DIR, "models")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# Cargar embeddings guardados
data = joblib.load(os.path.join(MODELS_DIR, "embeddings.joblib"))
X = data["embeddings"]    # shape (N, D)
y = data["y"]
class_names = data.get("class_names", None)

print("Loaded embeddings:", X.shape)
print("Labels:", y.shape)


# Celda 4 — Preparación: codificar etiquetas y semilla 

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_enc = le.fit_transform(y)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)


# Celda 5 — Definir el objective para Optuna

In [ ]:
# --- Celda A: Precompute StratifiedKFold splits and DMatrix objects ---
import os, joblib, numpy as np
from sklearn.model_selection import StratifiedKFold
import xgboost as xgb

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
RESULTS_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# y_enc y X deben venir de cargar embeddings joblib (ya tienes)
# skf config
N_FOLDS = 3          # CV interno en Optuna (rápido); luego puedes usar 5 para la evaluación final
RANDOM_SEED = 42

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
folds = []
for train_idx, val_idx in skf.split(X, y_enc):
    dtrain = xgb.DMatrix(X[train_idx], label=y_enc[train_idx])
    dval   = xgb.DMatrix(X[val_idx],   label=y_enc[val_idx])
    folds.append((dtrain, dval, train_idx, val_idx))

print(f"Precomputed {len(folds)} folds (DMatrix ready).")


In [ ]:
# --- Celda B: objective using precomputed DMatrix (GPU forced, no gpu_id) ---
import numpy as np
from sklearn.metrics import accuracy_score

def objective_precomputed(trial):
    # Hyperparam search space (moderado para pruebas)
    param = {
        "verbosity": 0,
        "objective": "multi:softprob",
        "num_class": int(len(np.unique(y_enc))),
        "booster": "gbtree",
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "eta": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 3.0),
        "alpha": trial.suggest_float("reg_alpha", 0.0, 2.0),
        "lambda": trial.suggest_float("reg_lambda", 0.0, 2.0),
        "tree_method": "hist",       # XGBoost ≥2.0 usa device='cuda' para GPU acceleration
        "device": "cuda",           # forzar uso GPU (si tu build lo soporta)
        "eval_metric": "mlogloss",
    }

    n_rounds = int(trial.suggest_int("n_estimators", 100, 800))  # rango moderado
    scores = []

    # iterate precomputed folds
    for (dtrain, dval, _, _) in folds:
        bst = xgb.train(
            params=param,
            dtrain=dtrain,
            num_boost_round=n_rounds,
            evals=[(dtrain, "train"), (dval, "valid")],
            early_stopping_rounds=15,
            verbose_eval=False
        )
        preds = np.argmax(bst.predict(dval), axis=1)
        scores.append(accuracy_score(dval.get_label(), preds))

    return float(np.mean(scores))


In [ ]:
%pip install xgboost==2.1.1 --extra-index-url https://pypi.nvidia.com

#  Celda 6 — Ejecutar el estudio de Optuna

In [ ]:
# --- Celda C: run Optuna study (quick) ---
import optuna, time, json, joblib

N_TRIALS = 25  # test rápido; si va bien sube a 25-50
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))

t0 = time.time()
study.optimize(objective_precomputed, n_trials=N_TRIALS, show_progress_bar=True)
t1 = time.time()

print("Optuna quick done in", round(t1-t0,1), "s")
print("Best CV value:", study.best_value)
print("Best params:", study.best_params)

# Save
joblib.dump(study, os.path.join(RESULTS_DIR, "optuna_precomputed_study.joblib"))
with open(os.path.join(RESULTS_DIR, "optuna_precomputed_best.json"), "w") as f:
    json.dump({"best_value": float(study.best_value), "best_params": study.best_params}, f, indent=2)


# Celda 7 — Visualización y resumen del estudio

In [ ]:
# --- Celda 7: resumen de resultados Optuna ---
import optuna.visualization as vis

print("Best value (CV acc):", study.best_value)
print("\nBest params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

# Gráficos interactivos (puedes abrirlos en navegador si usas VSCode)
try:
    display(vis.plot_optimization_history(study))
    display(vis.plot_param_importances(study))
except Exception as e:
    print("Visualización no disponible:", e)


# Celda 8 — Guardar los mejores hiperparámetros para usar en el modelo final

In [ ]:
# --- Celda 8: guardar mejores parámetros ---
best_params = study.best_params

# Guardar JSON en carpeta de resultados
best_params_path = os.path.join(RESULTS_DIR, "best_xgb_params.json")
with open(best_params_path, "w") as f:
    json.dump(best_params, f, indent=2)

print(f"✅ Mejores parámetros guardados en {best_params_path}")


# Celda 9 — (opcional) Validación final rápida

In [ ]:
# --- Celda 9: validación cruzada final con mejores parámetros ---
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
scores = []

for train_idx, val_idx in skf.split(X, y_enc):
    dtrain = xgb.DMatrix(X[train_idx], label=y_enc[train_idx])
    dval = xgb.DMatrix(X[val_idx], label=y_enc[val_idx])
    
    bst = xgb.train(
        params={**best_params, "objective": "multi:softprob", "num_class": len(np.unique(y_enc)),
                "tree_method": "hist", "device": "cuda"},
        dtrain=dtrain,
        num_boost_round=best_params.get("n_estimators", 200),
        evals=[(dval, "val")],
        early_stopping_rounds=15,
        verbose_eval=False
    )
    
    preds = np.argmax(bst.predict(dval), axis=1)
    scores.append(accuracy_score(y_enc[val_idx], preds))

print("✅ Mean CV accuracy:", np.mean(scores))
